# 01 — EDA: Smart CI Test Prioritization (LRTS)

Thin notebook: every number and figure is produced by `src/stp/eda/*` (each section writes `reports/figures/*.png` and `reports/eda_json/*.json`). Prerequisites: `python -m stp.data.download` and `python -m stp.data.build_tables`. The written findings are in `reports/EDA_REPORT.md`.

In [ ]:
import json
from IPython.display import Image, display, Markdown
import pandas as pd
from stp import paths
from stp.eda import a_baselines, b_costs, c_shape, d_failures, e_changes, f_signal, g_ttff, h_split
pd.set_option('display.width', 200)

def show(*names):
    for n in names:
        display(Image(filename=str(paths.FIGURES / f'{n}.png')))

## Audits
Count cross-checks (see `src/stp/data/audit.py`).

In [ ]:
audit = json.loads((paths.EDA_JSON / 'audit.json').read_text())
display(pd.DataFrame(audit['test_files']['pass_fail_count_match']))
display(pd.DataFrame(audit['first_time_failure']['per_project']))
display(pd.DataFrame(audit['changes']['vs_change_stats_csv']))

## A. Motivation: paper baselines (APFDc, macro over projects)

In [ ]:
s = a_baselines.run()
pd.DataFrame({v: {t: s[v]['mean_apfdc'][t]['macro'] for t in a_baselines.TECHNIQUES} for v in ['All','DeConf','FirstFail']}).round(3)

In [ ]:
show('A_headline_all_vs_firstfail', 'A_macro_table')

## B. The problem is real: costs

In [ ]:
s = b_costs.run()
pd.DataFrame({k: s['full'][k]['per_project'] for k in ['duration_h_median','share_runs_over_1h','share_runs_over_2h','share_runs_over_6h','classes_per_run_median']}).round(3)

In [ ]:
show('B_duration_cdf', 'B_classes_per_run')

## C. Dataset shape

In [ ]:
s = c_shape.run()
pd.DataFrame(s['per_project'])

In [ ]:
show('C_runs_per_project', 'C_builds_per_month')

## D. Failure structure

In [ ]:
s = d_failures.run()
pd.DataFrame({k: s[k]['per_project'] for k in ['ftf_share_of_failures','ftf_share_of_failures_after_burn_in','failed_runs_with_any_ftf','failed_runs_all_ftf']}).round(3)

In [ ]:
show('D_first_time_failures', 'D_failure_frequency_tail', 'D_failed_classes_per_run', 'D_flaky_share')

## E. Change data

In [ ]:
s = e_changes.run()
pd.DataFrame({k: s[k]['per_project'] for k in ['files_per_build_median','lines_per_build_median','share_builds_with_patch_text','share_changed_files_with_patch']}).round(3)

In [ ]:
show('E_change_size', 'E_extensions')

## F. Is there a change signal? (token overlap lift)

In [ ]:
s = f_signal.run()
pd.DataFrame({k: s[k]['per_project'] for k in ['lift_ftf_k1','lift_ftf_k2','lift_ftf_k3','lift_fail_k1','recall_ftf_k1','share_rows_overlap_ge1']}).round(3)

In [ ]:
show('F_overlap_lift', 'F_overlap_share')

## G. Time to first failure (SAMPLE of <=300 failed runs/project)

In [ ]:
s = g_ttff.run()
pd.DataFrame({k: s['all_failed_runs'][k]['per_project'] for k in ['median_min_Random','median_min_QTF','median_min_LastFail','median_min_Oracle','median_min_suite_s']}).round(1)

In [ ]:
show('G_time_to_first_failure')

## H. Split feasibility (chronological 70/15/15)

In [ ]:
s = h_split.run()
pd.DataFrame(s['table']).set_index(['project','split'])

In [ ]:
show('H_split_ftf_runs')